# PromptAlignmentMetric

## What it measures

Whether the output obeys a list of explicit instructions you supply as
`prompt_instructions`. The judge checks each instruction independently and scores the
proportion satisfied. Unlike every other metric here it takes a **constructor argument
that is part of the test definition** - the instructions are the specification being
tested against.

## When it is useful

When a downstream consumer depends on the *shape* and *discipline* of an AI response, not
just its content. Here the frontend renders `risk_level` as a badge, the decisions
endpoint accepts only four action values, and the whole audit story rests on every claim in
the rationale carrying a resolvable `C#`/`T#` evidence label. An answer that is factually
excellent but breaks any of those is still a defect.

It is also the natural metric for testing a *negative* instruction - something the output
must not do - which is otherwise awkward to assert.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `LLMTestCase` |
| `input` | yes |
| `actual_output` | yes |
| `prompt_instructions` (constructor) | yes - a list of instruction strings |

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
EXPECTED_SCHEMA_VERSION = "1.0.0"

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    if served and served != EXPECTED_SCHEMA_VERSION:
        print(f"WARNING: application reports contract version {served}, these "
              f"notebooks were written against {EXPECTED_SCHEMA_VERSION}. "
              f"Field names may have changed - see docs/evaluation-contract.md.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoint exercised

`POST /api/cases/{case_id}/investigate` on **s2, "High-risk jurisdiction transfer"** - a
scenario rich enough that the rationale must cite both retrieved policy chunks and tool
results, which is what makes the evidence-labelling instruction meaningful to test.

An important scoping note. This notebook does **not** read the application's prompt: prompt
bodies are never stored or served, only a `prompt_version` identifier. The instructions
below are therefore derived from the *published response contract* - the enumerations and
guarantees the API documents to its consumers - not from the internal prompt text. That is
the correct thing for a black-box acceptance test to hold the application to, and it is
also stricter in the useful direction: it tests the promise made to callers rather than the
wording used internally.

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

In [ ]:
# --------------------------------------------------------------------------
# The exact request.
# --------------------------------------------------------------------------
CASE_ID = SCENARIOS["s2"]["case_id"]      # "High-risk jurisdiction transfer" scenario

print("POST", f"{API_BASE}/api/cases/{CASE_ID}/investigate")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
print("body: (none - this endpoint reads no request body)")

In [ ]:
# --------------------------------------------------------------------------
# The raw response.
# --------------------------------------------------------------------------
investigation = api("POST", f"/api/cases/{CASE_ID}/investigate", expect_status=201)

show("POST /api/cases/{case_id}/investigate",
     {k: v for k, v in investigation.items() if k != "retrieved_context"})
print()
print(f"prompt_version reported by the application: {investigation['prompt_version']!r}")
print("(the prompt body itself is never stored or served - only this identifier)")

## Mapping the API response onto DeepEval fields

| DeepEval field | Source | Note |
|---|---|---|
| `input` | the investigation task statement | Matches the `input` the application's own eval export uses for investigation runs |
| `actual_output` | the assessment, serialized | `risk_level`, `recommended_action`, `rationale`, `confidence` plus the resolved evidence labels |
| `prompt_instructions` | derived below from the published contract | The specification under test |

`actual_output` is assembled rather than taken from one field because the instructions span
several fields at once: the enumerations live in three separate keys and the labelling
requirement links `rationale` to `evidence`. Serializing them together lets the judge see
the whole obligation. The application's own `GET /api/eval/export/{run_id}` flattens
investigation output the same way.

In [ ]:
# --------------------------------------------------------------------------
# Deriving the instructions.
#
# Each one restates a guarantee the API publishes about this endpoint's
# response, so the test holds the application to its own documented contract
# rather than to a preference invented here.
#
# The final instruction is a negative: the platform's core safety property is
# that no AI-producing endpoint can finalise a case, and the response must
# present itself as a recommendation. Only POST /api/cases/{id}/decisions can
# move a case to a terminal status; that invariant is asserted directly below,
# independently of the judge.
# --------------------------------------------------------------------------
PROMPT_INSTRUCTIONS = [
    "State a risk level that is exactly one of: low, medium, high.",
    "State a recommended action that is exactly one of: approve, reject, escalate, "
    "request_evidence.",
    "Support every claim made in the rationale with an evidence label of the form "
    "C<number> for a retrieved chunk or T<number> for a tool call.",
    "State a confidence that is exactly one of: high, medium, low.",
    "Present the outcome as a recommendation for an analyst to act on; do not state or "
    "imply that the case has been decided, closed, approved or rejected.",
]

for n, instruction in enumerate(PROMPT_INSTRUCTIONS, start=1):
    print(f"{n}. {textwrap.fill(instruction, width=92, subsequent_indent='   ')}")

# Deterministic cross-check of the same contract, independent of the judge.
import re

ENUMS = {
    "risk_level": {"low", "medium", "high"},
    "recommended_action": {"approve", "reject", "escalate", "request_evidence"},
    "confidence": {"high", "medium", "low"},
}
print()
for field, allowed in ENUMS.items():
    value = investigation[field]
    print(f"  {field:<18} = {value!r:<18} in {sorted(allowed)} -> "
          f"{'OK' if value in allowed else 'CONTRACT VIOLATION'}")

labels_in_rationale = set(re.findall(r"\b([CT]\d+)\b", investigation["rationale"]))
labels_resolved = {ref["label"] for ref in investigation["evidence"]}
print(f"  labels cited in rationale : {sorted(labels_in_rationale)}")
print(f"  labels resolved in evidence: {sorted(labels_resolved)}")
print(f"  unresolved labels          : {sorted(labels_in_rationale - labels_resolved)}")

# The safety invariant: a recommendation must not have become a decision.
case_after = api("GET", f"/api/cases/{CASE_ID}")
print(f"  case status after investigate: {case_after['status']!r} "
      f"(decisions recorded: {len(case_after['decisions'])})")
if case_after["status"] not in ("open", "in_review"):
    raise RuntimeError(
        f"Case {CASE_ID} left a working status after an investigation run. The platform's "
        f"core invariant is that only POST /api/cases/{{id}}/decisions can finalise a "
        f"case; this is a serious regression, not a scoring nuance."
    )

In [ ]:
# --------------------------------------------------------------------------
# Build the test case and print each DeepEval role explicitly.
# --------------------------------------------------------------------------
from deepeval.test_case import LLMTestCase

INVESTIGATION_TASK = (
    f"Investigate case {CASE_ID} for AML/KYC risk and recommend an action "
    f"(approve, reject, escalate or request_evidence)."
)

assessment = {
    "risk_level": investigation["risk_level"],
    "recommended_action": investigation["recommended_action"],
    "rationale": investigation["rationale"],
    "confidence": investigation["confidence"],
    "evidence": [{"label": ref["label"], "kind": ref["kind"]}
                 for ref in investigation["evidence"]],
}
ACTUAL_OUTPUT = json.dumps(assessment, indent=2)

test_case = LLMTestCase(input=INVESTIGATION_TASK, actual_output=ACTUAL_OUTPUT)

print("USER INPUT")
print(" ", test_case.input)
print()
print("ACTUAL OUTPUT (serialized assessment)")
print(textwrap.indent(test_case.actual_output, "  "))
print()
print("PROMPT INSTRUCTIONS (the specification under test)")
for n, instruction in enumerate(PROMPT_INSTRUCTIONS, start=1):
    print(f"  {n}. {instruction}")
print()
print("EXPECTED OUTPUT (golden) : not used by this metric")

## Judge and threshold

- **Judge model**: `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `PromptAlignmentMetric`.

The score is the proportion of instructions satisfied, so with five instructions the
granularity is `0.2`. `0.5` therefore means "at least three of five obeyed".

That is a weak bar for a contract test, and deliberately so here: the default is kept so
the notebook demonstrates the documented configuration. Three of the five instructions are
also checked *deterministically* in the derivation cell above, which is the stronger gate -
an enumeration violation raises there regardless of what the judge decides. Use this metric
for the instructions that cannot be regex-checked (the labelling discipline and the
recommendation-not-decision framing) and keep the deterministic assertions for the rest.

In [ ]:
from deepeval.metrics import PromptAlignmentMetric

metric = PromptAlignmentMetric(
    prompt_instructions=PROMPT_INSTRUCTIONS,
    threshold=0.5,          # DeepEval's documented default
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

## Limitations in a black-box acceptance test

1. **The instructions are a reconstruction.** The application never serves its prompt
   bodies, only `prompt_version`. These instructions restate the published response
   contract, which is what callers depend on - but a genuine prompt regression that does
   not break the contract will not be caught here.
2. **Score granularity is coarse.** With five instructions, one violation costs `0.2`.
   Adding instructions to gain resolution changes the denominator and makes runs
   incomparable with earlier ones.
3. **The application already enforces most of this server-side.** Synthesis validates the
   enumerations and resolves every `C#`/`T#` label, failing with `502 llm_response_invalid`
   before a bad response can reach the API surface. So a passing score here largely
   confirms the *validator* works. That is worth knowing, but it is not the same as
   confirming the model is well-behaved - the interesting failures are the ones validation
   would reject, and those never become an `actual_output` to score.
4. **The negative instruction is judged softly.** "Do not imply the case is decided" is a
   tone judgement. The hard version of that check is the case-status assertion in the
   derivation cell, which reads the case back and fails on any terminal status.
5. **Instructions are scored independently.** The metric cannot express "instruction 3
   only applies when the rationale makes a factual claim", so conditional obligations have
   to be flattened into unconditional ones, which slightly overstates what is being
   tested.